In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [5]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. **Install Ollama** from [https://ollama.com/download](https://ollama.com/download)
   - **macOS**: download the `.pkg` and install it
   - **Windows**: download the `.msi` and install it
   - **Linux**: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally** by opening a terminal and running:
   ```bash
   ollama run llama3
   ```
   This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

3. **Test the local server** with:
   ```bash
   curl http://localhost:11434
   ```
   You should get a response like:
   ```json
   {"models": [...]}  
   ```

4. **Optional: use Python**
   ```bash
   pip install ollama
   ```
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": your_prompt}]
   )

   print(response['message']['content'])
   ```


In [6]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

The FAQ context doesn’t mention “Olama” specifically, but it does say you can run the course locally instead of using Codespaces.

To run it locally, you should be comfortable setting up:

- Python
- `uv`
- Jupyter
- Docker
- any other tools needed for the module

If you run locally, make sure you:

- document your setup
- keep your environment reproducible

If you meant a local model/tool setup related to the course, the context doesn’t provide specific steps for that.


In [7]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Maybe — it depends on the course’s enrollment rules and whether it’s still open.\n\nIf you want, send me:\n- the course name,\n- where it’s offered,\n- and whether you want help contacting the instructor or checking the enrollment deadline,\n\nand I can help you figure out the best next step or draft a message asking to join.'

In [8]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [10]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [11]:
len(response.output)

1

In [12]:
call = response.output[0]

In [13]:
call

ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment eligibility late registration open enrollment"}', call_id='call_nuJS2PpgBgCkxC7nMJVH0qZl', name='search', type='function_call', id='fc_0a2805d3278edd88006a398df5535c819b9051c09a2151b3c4', namespace=None, status='completed')

In [14]:
import json

args = json.loads(call.arguments)
args

{'query': 'Can I join the course if I just discovered it? enrollment eligibility late registration open enrollment'}

In [15]:
call.name

'search'

In [16]:
results = search(**args)

In [17]:
result_json = json.dumps(results, indent=2)
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "977bf7786c",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?",
    "answer": "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."
  },
  {
    "id": "aa310de435",
    "course": "llm-zoomcamp",
    "section": "Module 1: RAG",
    "question": "Can I run the course locally instead of Codespaces?",
    "answer": "Ye

In [18]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [19]:
messages.append(call)
print(messages)

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}, ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment eligibility late registration open enrollment"}', call_id='call_nuJS2PpgBgCkxC7nMJVH0qZl', name='search', type='function_call', id='fc_0a2805d3278edd88006a398df5535c819b9051c09a2151b3c4', namespace=None, status='completed')]


In [20]:
messages.append(function_call_output)

In [21]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment eligibility late registration open enrollment"}', call_id='call_nuJS2PpgBgCkxC7nMJVH0qZl', name='search', type='function_call', id='fc_0a2805d3278edd88006a398df5535c819b9051c09a2151b3c4', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_nuJS2PpgBgCkxC7nMJVH0qZl',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LL

In [22]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [23]:
print(response.output_text)

Yes — you can still join the course.

If you want to receive a certificate, make sure you submit your project while submissions are still open.


In [24]:
usage = response.usage
usage

ResponseUsage(input_tokens=799, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=33, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=832)

In [25]:
usage.input_tokens, usage.output_tokens

(799, 33)

In [26]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0001176


In [27]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [28]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [29]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [30]:
for item in response.output:
    print(item)

ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment registration late join"}', call_id='call_oUMM3ZS5SiYtWN2ft3FXJ3a3', name='search', type='function_call', id='fc_09ef68d64db7f542006a398df7a2748198996a4f709cebca17', namespace=None, status='completed')
ResponseFunctionToolCall(arguments='{"query":"new student discovered course can I join course FAQ enrollment"}', call_id='call_6IzUYIW06rYd0FmEzht6kAlG', name='search', type='function_call', id='fc_09ef68d64db7f542006a398df7a2e08198b2139854165445f9', namespace=None, status='completed')
ResponseFunctionToolCall(arguments='{"query":"course access join after start late enrollment FAQ"}', call_id='call_gte2cr6Q2RVRAEh2GmV7Y4R9', name='search', type='function_call', id='fc_09ef68d64db7f542006a398df7a2e88198a7ff188f36ada22a', namespace=None, status='completed')


In [31]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment registration late join"}
function_call: search {"query":"new student discovered course can I join course FAQ enrollment"}
function_call: search {"query":"course access join after start late enrollment FAQ"}


In [32]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment registration late join"}', call_id='call_oUMM3ZS5SiYtWN2ft3FXJ3a3', name='search', type='function_call', id='fc_09ef68d64db7f542006a398df7a2748198996a4f709cebca17', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"new student discovered course can I join course FAQ enrollment"}'

In [33]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course discovered course can I join enroll registration late join FAQ"}
function_call: search {"query":"course enrollment late join new student discovered course FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open. You can also start learning and doing the materials even if you discovered it late.

If you want, I can also help with how to get started or explain the certificate requirements.


In [34]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [35]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [36]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join the course enrollment discovered course can I join late registration FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

One important note: if you want a certificate, you’ll need to submit your project while submissions are still open.

If you’d like, I can also help you find the course materials or explain how to start from where you are now.


In [37]:
result

'Yes — you can still join the course.\n\nOne important note: if you want a certificate, you’ll need to submit your project while submissions are still open.\n\nIf you’d like, I can also help you find the course materials or explain how to start from where you are now.'

In [38]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit opening chess course FAQ"}
iteration #3...
function_call: search {"query":"gambit chess opening FAQ"}
iteration #4...
ASSISTANT:
I couldn’t find a course-related FAQ entry for “queen gambit,” and it looks like that’s probably not a course topic.

If you meant something specific from the course, feel free to rephrase it and I’ll check the FAQ again. Are there other areas you want to explore?


In [39]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [40]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [41]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [42]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [43]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [44]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [45]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [46]:
result = runner.loop(
    prompt='How do I run Olama locally?',
    callback=callback,
)

-> Response received


-> Response received


In [47]:
result.cost

CostInfo(input_cost=Decimal('0.00113775'), output_cost=Decimal('0.0011925'), total_cost=Decimal('0.00233025'))

In [48]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama run locally Ollama locally"}', call_id='call_2929gt

In [49]:
result2 = runner.loop(
    prompt='How do I run a different model?',
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [50]:
runner.run();

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.
